#### 1.原始訊息
##### 收到日期：20260813
##### 完成日期：20260817
##### 花費時間：4 Hr
需求：以加權指數自 2000/1/1 起日資料，當日收盤價若向上穿越過季線（60日），則買入；反之，則賣出。請統計出以下幾項資料：

- ⬤ 總損益、交易次數、勝率、平均賺賠比、最大連續虧損
- ⬤ 最後持有部位資訊（進場日、多空、價位）
- ⬤ （進階）統計所有出場交易之損益、持有時間

P.S：
- 1）先以金融市場習慣說法為主，查了若仍不懂請勇於提問。利用每次機會知道對方的邏輯
- 2）本策略一旦進場後，非多即空，不會空手
- 3）可能是題組，請考量日後調整彈性
- 4）不限制使用工具（個人偏好EXCEL、GOOGLE SPREADSHEET）

#### 2.策略及定義假設
- **基本設定**
  - 標的：台灣加權指數（TAIEX / Y9999）
  - 資料頻率：日資料
  - 回測起始日：2000/01/01
  - 季線：60 日簡單移動平均線（SMA60）

- **交易規則**
  - 做多
    - 昨日收盤價 ≤ 昨日 SMA60
    - 今日收盤價 > 今日 SMA60
    - 視為**向上穿越季線**，建立多頭部位（Long）
  - 做空
    - 昨日收盤價 ≥ 昨日 SMA60
    - 今日收盤價 < 今日 SMA60
    - 視為**向下穿越季線**，建立空頭部位（Short）
  - 部位規則
    - 策略開始持有部位後，維持**非多即空**
    - 不存在空手狀態
    - 出現反向訊號時：
      1. 將原有部位平倉
      2. 同時建立反向部位

- **成交假設**
  - 訊號以**當日收盤價**判斷
  - 假設以**訊號當日收盤價成交**

- **損益定義**
  - 損益先以**加權指數點數**表示
  - 暫不考慮：
    - 初始本金
    - 每點價值
    - 交易成本
    - 手續費
    - 稅費
    - 滑價
      - 原本預期成交的價格，和實際真正成交的價格之間的差距。
      - 例如你看到加權指數在 20,000 點出現買進訊號，理論上希望用 20,000 點成交，但實際下單後可能成交在 20,005 點，這多出來的 5 點就是滑價。
      - 滑價常見原因包括市場快速波動、流動性不足、買賣價差，以及從訊號產生到訂單真正進市場之間的時間差。

- **交易定義**
  - 一筆完整交易定義為：
    - **進場 → 持有 → 出場**
  - 反手時：
    - 原部位完成一筆交易
    - 同時開始下一筆反向交易
  - 尚未出場的最後一筆部位視為**未平倉部位（Open Position）**

- **持有時間**
  - 持有時間以**交易日（Trading Days）**計算
  - 定義：
    - 今日收盤進場，下一個交易日收盤出場，持有時間為 1 個交易日
  - 不使用日曆日計算，因此週末及休市日不額外計入持有時間

- **績效指標**
  - 總損益
    - 所有已完成交易之損益點數加總
  - 交易次數
    - 完成「進場 → 出場」的交易筆數
    - 未平倉的最後部位不列入已完成交易次數
  - 勝率
    - 獲利交易次數占全部已完成交易次數的比例
  - 平均賺賠比
    - 平均獲利交易的獲利點數，相對於平均虧損交易虧損點數絕對值的比例
  - 最大連續虧損
    - 歷史交易紀錄中，連續出現虧損交易的最大筆數

- **最後持有部位**
  - 紀錄回測資料截止日仍持有之部位：
    - 進場日
    - 多空方向（Long / Short）
    - 進場價位

- **出場交易明細**
  - 每筆已完成交易至少紀錄：
    - 進場日
    - 出場日
    - 多空方向
    - 進場價
    - 出場價
    - 損益點數
    - 持有交易日數

- **第一筆部位如何建立**
  - 等待回測起始日後第一次穿越訊號再進場

##### 3.程式內容

載入資料

In [2]:
from pathlib import Path
import pandas as pd
file_path = Path("../../../input/Y9999_20260813.xlsx")

df = pd.read_excel(
    file_path,
    header=1
)


清洗資料

In [3]:
df = df.rename(columns={
    "年月日": "trade_date",
    "收盤價(元)": "close_price"
})
df = df[["trade_date", "close_price"]]
df = df.sort_values("trade_date").reset_index(drop=True)
df["trade_date"] = pd.to_datetime(
    df["trade_date"],
    errors="coerce"
)

建立 sma

In [4]:
df["sma60"] = df["close_price"].rolling(60).mean()

建立訊號
- signal : Enum
    - "Long"
    - "Short"
    - "-"

In [5]:
def get_signal(index):
    currValue = df.loc[index, "close_price"]
    currSma = df.loc[index, "sma60"]

    yestValue = df.loc[index - 1, "close_price"]
    yestSma = df.loc[index - 1, "sma60"]

    if currValue > currSma and yestValue <= yestSma:
        # print(index, "L")
        return "Long"

    elif currValue < currSma and yestValue >= yestSma:
        # print(index, "S")
        return "Short"
    else:
        return "-"

df["signal"] = "-"
for index in  range(60, len(df)):
    df.loc[index, "signal"] = get_signal(index)

獨立訊號為表並取 2000-1-1 後

In [6]:
signal_df = df.loc[
    (df["signal"] != "-") & (df["trade_date"] >= pd.Timestamp("2000-01-01")),
    ["trade_date", "close_price", "sma60", "signal"]
].copy()
signal_df = signal_df.reset_index(drop=True)

建立單次訊號損益及持有天數

In [7]:
def get_profit(index):
    currValue = signal_df.loc[index, "close_price"]
    yestValue = signal_df.loc[index - 1, "close_price"]
    isLong = signal_df.loc[index -1, "signal"] == "Long"
    
    if(isLong):
        return currValue - yestValue
    else: 
        return yestValue - currValue
    
def get_hold_day(index):
    currDay = signal_df.loc[index, "trade_date"]
    yestDay = signal_df.loc[index - 1, "trade_date"]
    hold_day = (currDay - yestDay).days
    return hold_day

def get_profit_pct(index):
    
    value = signal_df.loc[index - 1, "close_price"]
    profit = signal_df.loc[index - 1, "profit"]
    return profit / value



for index in  range(1, len(signal_df)):
    signal_df.loc[index - 1, "profit"] = get_profit(index)
    signal_df.loc[index - 1, "profit_pct"] = get_profit_pct(index)
    signal_df.loc[index -1, "hold_day"] = get_hold_day(index)


取得統計資訊
- 總損益：sum_profit
- 勝率：win_rate
- 平均賺賠比：profit_mean_ratio
- 最大連續虧損(次數)：loss_continue_max

In [8]:
sum_profit = 0
total_profit_pct = 1

win_count = 0
loss_count = 0

long_win_count = 0 
long_loss_count = 0
short_win_count = 0
shrot_loss_count = 0

win_profit_sum = 0
loss_profit_sum = 0

loss_continue_max = 0
curr_loss_continue_count = 0

loss_continue_max_price = 0.0
curr_loss_continue_price = 0.0

long_profit_pct_sum = 1.0
short_profit_pct_sum = 1.0

def set_ls_win_count(profit, isLong:bool):
    global long_win_count 
    global long_loss_count 
    global short_win_count 
    global shrot_loss_count 
    if(profit>0):
        if(isLong):
            long_win_count +=1
        else : 
            short_win_count +=1
    if(profit<0):
        if(isLong):
            long_loss_count +=1
        else:
            shrot_loss_count +=1
            
        


def set_loss_coutinue_max(profit):
    is_profit_win = profit > 0 
    global curr_loss_continue_count
    global loss_continue_max
    
    global loss_continue_max_price
    global curr_loss_continue_price
    global long_profit_pct_sum
    global short_profit_pct_sum
        
        
    if(is_profit_win):
        if(curr_loss_continue_count > loss_continue_max):
            loss_continue_max = curr_loss_continue_count
            
        if(loss_continue_max_price > curr_loss_continue_price):
            loss_continue_max_price = curr_loss_continue_price
        curr_loss_continue_count = 0
        curr_loss_continue_price = 0
    else:
        curr_loss_continue_count = curr_loss_continue_count + 1
        curr_loss_continue_price = curr_loss_continue_price + profit 
    

for index in range(0, len(signal_df) - 1):
    profit = signal_df.loc[index,"profit"]
    profit_pct = signal_df.loc[index,"profit_pct"]
    
    isLong = signal_df.loc[index,"signal"] == "Long"
    
    sum_profit = sum_profit+profit
    total_profit_pct = total_profit_pct * (1 + profit_pct)
    if(isLong):
        long_profit_pct_sum = long_profit_pct_sum * (1 + profit_pct)
    else: 
        short_profit_pct_sum = short_profit_pct_sum * (1 + profit_pct)
        
    
    set_ls_win_count(profit, isLong)
    set_loss_coutinue_max(profit)
    
    if(profit > 0):
        win_count = win_count + 1
        win_profit_sum = win_profit_sum + profit
    else:
        loss_count = loss_count + 1
        loss_profit_sum = loss_profit_sum + profit
        
total_count = win_count + loss_count
if total_count > 0:
    win_rate = win_count / total_count
else:
    win_rate = 0
    
win_profit_mean = win_profit_sum / win_count
loss_profit_mean = loss_profit_sum / loss_count


profit_mean_ratio = abs(win_profit_mean) / abs(loss_profit_mean)
print("-" * 10, "統計資訊", "-" * 10)
print("總損益",f"{sum_profit:.2f}")
print("總損益率",f"{total_profit_pct*100:.2f}%")
print("交易次數",len(signal_df))
print("勝率",f"{win_rate*100:.2f}%")
print("平均賺賠比", f"{profit_mean_ratio:.2f}")
print("最大連續虧損(次數)", loss_continue_max)
print("最大連續虧損(點數)", f"{loss_continue_max_price:.2f}")
print("-" * 10, "最後持有部位資訊", "-" * 10)

print("進場日", signal_df.loc[len(signal_df)-1, "trade_date"])
print("多空(Long/Short)", signal_df.loc[len(signal_df)-1, "signal"])
print("價位", signal_df.loc[len(signal_df)-1, "close_price"])


print("-" * 10, "其它資訊", "-" * 10)
print("最高持有日",signal_df["hold_day"].max())
print("單次最高獲利",f"{signal_df["profit"].max():.2f}")
print("單次最高虧損",f"{signal_df["profit"].min():.2f}")

print("只做多損益率",f"{long_profit_pct_sum*100:.2f}%")
print("只做空損益率",f"{short_profit_pct_sum*100:.2f}%")

long_win_pct = long_win_count/(long_win_count+long_loss_count)
short_win_pct = short_win_count/(short_win_count+shrot_loss_count)

print("只做多勝率",f"{long_win_pct*100:.2f}%")
print("只做空勝率",f"{short_win_pct*100:.2f}%")




---------- 統計資訊 ----------
總損益 32879.93
總損益率 1473.27%
交易次數 334
勝率 23.42%
平均賺賠比 6.03
最大連續虧損(次數) 24
最大連續虧損(點數) -6415.29
---------- 最後持有部位資訊 ----------
進場日 2026-08-10 00:00:00
多空(Long/Short) Long
價位 44928.76
---------- 其它資訊 ----------
最高持有日 222.0
單次最高獲利 9496.45
單次最高虧損 -1561.60
只做多損益率 1232.33%
只做空損益率 119.55%
只做多勝率 29.52%
只做空勝率 17.37%


In [9]:
def set_single_max(index):
    yest_signal_date = signal_df.loc[index - 1, "trade_date"]
    curr_signal_date = signal_df.loc[index, "trade_date"]

    entry_price = signal_df.loc[index - 1, "close_price"]
    signal = signal_df.loc[index - 1, "signal"]

    max_period_price = df.loc[
        (df["trade_date"] >= yest_signal_date) &
        (df["trade_date"] <= curr_signal_date),
        "close_price"
    ].max()

    min_period_price = df.loc[
        (df["trade_date"] >= yest_signal_date) &
        (df["trade_date"] <= curr_signal_date),
        "close_price"
    ].min()

    if signal == "Long":
        max_period_profit = max_period_price - entry_price
        min_period_profit = min_period_price - entry_price

    elif signal == "Short":
        max_period_profit = entry_price - min_period_price
        min_period_profit = entry_price - max_period_price

    signal_df.loc[index - 1, "max_period_price"] = max_period_price
    signal_df.loc[index - 1, "min_period_price"] = min_period_price

    signal_df.loc[index - 1, "max_period_profit"] = max_period_profit
    signal_df.loc[index - 1, "min_period_profit"] = min_period_profit

    signal_df.loc[index - 1, "max_period_profit_pct"] = max_period_profit / entry_price
    signal_df.loc[index - 1, "min_period_profit_pct"] = min_period_profit / entry_price
    
    
    

for index in range(1, len(signal_df)):
    set_single_max(index)

period_max = signal_df.loc[signal_df["max_period_profit_pct"].idxmax()]
period_min = signal_df.loc[signal_df["min_period_profit_pct"].idxmin()]

profit_pct_max = signal_df.loc[signal_df["profit_pct"].idxmax()]
profit_pct_min = signal_df.loc[signal_df["profit_pct"].idxmin()]


print("-"*10,"單次訊號波動最高值","-"*10,)
print(period_max)
print("-"*10,"單次訊號波動最低值","-"*10,)
print(period_min)


print("-"*10,"單次訊號最高收益","-"*10,)
print(profit_pct_max)
print("-"*10,"單次訊號最低收益","-"*10,)
print(profit_pct_min)

---------- 單次訊號波動最高值 ----------
trade_date               2001-11-05 00:00:00
close_price                          4080.51
sma60                               4068.742
signal                                  Long
profit                               1787.32
profit_pct                          0.438014
hold_day                               178.0
max_period_price                      6462.3
min_period_price                     4080.51
max_period_profit                    2381.79
min_period_profit                        0.0
max_period_profit_pct               0.583699
min_period_profit_pct                    0.0
Name: 7, dtype: object
---------- 單次訊號波動最低值 ----------
trade_date               2000-03-13 00:00:00
close_price                          8811.95
sma60                              9190.8465
signal                                 Short
profit                               -721.92
profit_pct                         -0.081925
hold_day                                10.0
max_period_pr

In [10]:
print("-" * 10, "統計所有出場交易之損益、持有時間", "-" * 10)
signal_df.rename(columns={
    "trade_date": "交易日期",
    "close_price": "收盤價",
    "sma60": "季線",
    "signal": "多空訊號",
    "profit": "損益點數",
    "profit_pct": "損益%",
    "hold_day": "持有交易日",
    "max_period_price": "持有期間最高價",
    "min_period_price": "持有期間最低價",
    "max_period_profit": "持有期間最大獲利點數",
    "min_period_profit": "持有期間最大虧損點數",
    "max_period_profit_pct": "持有期間最大獲利%",
    "min_period_profit_pct": "持有期間最大虧損%"
}, inplace=True)
signal_df

---------- 統計所有出場交易之損益、持有時間 ----------


,交易日期,收盤價,季線,多空訊號,損益點數,損益%,持有交易日,持有期間最高價,持有期間最低價,持有期間最大獲利點數,持有期間最大虧損點數,持有期間最大獲利%,持有期間最大虧損%
0,2000-03-13,8811.95,9190.846500,Short,-721.92,-0.081925,10.0,9533.87,8536.05,275.90,-721.92,0.031310,-0.081925
1,2000-03-23,9533.87,9330.423000,Long,-159.26,-0.016705,22.0,10186.17,9374.61,652.30,-159.26,0.068419,-0.016705
2,2000-04-14,9374.61,9602.798500,Short,259.14,0.027643,54.0,9374.61,8349.91,1024.70,0.00,0.109306,0.000000
3,2000-06-07,9115.47,9088.918167,Long,-47.59,-0.005221,1.0,9115.47,9067.88,0.00,-47.59,0.000000,-0.005221
4,2000-06-08,9067.88,9089.974833,Short,3691.76,0.407125,215.0,9067.88,4614.63,4453.25,0.00,0.491102,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
329,2026-07-21,44232.87,43709.963000,Long,-578.03,-0.013068,3.0,44850.81,43654.84,617.94,-578.03,0.013970,-0.013068
330,2026-07-24,43654.84,43964.307500,Short,-956.76,-0.021916,12.0,44611.60,39933.30,3721.54,-956.76,0.085249,-0.021916
331,2026-08-05,44611.60,44189.595167,Long,-385.69,-0.008646,2.0,44611.60,44225.91,0.00,-385.69,0.000000,-0.008646
332,2026-08-07,44225.91,44278.758333,Short,-702.85,-0.015892,3.0,44928.76,44225.91,0.00,-702.85,0.000000,-0.015892
